In [0]:
#Check Spark version and active catalog
print(f"Spark version: {spark.version}")

#Display the current catalog and schema
spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
""").show()

In [0]:
#Create a small test DataFrame
sample_data = [
    (1, "Sudoku Easy"),
    (2, "Sudoku Medium"),
    (3, "Sudoku Hard"),
]

sample_df = spark.createDataFrame(
    sample_data,
    ["id", "difficulty"]
)

display(sample_df)

In [0]:
#Create schemas for the Medallion Architecture
schemas = [
    "sudoku_bronze",
    "sudoku_silver",
    "sudoku_gold",
]

for schema_name in schemas:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS workspace.{schema_name}"
    )

    print(f"Schema created or verified: workspace.{schema_name}")

In [0]:
#List available schemas in the workspace catalog
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

#Notes:
    #Bronze (Raw Data): store ingested raw data
    #Silver (Cleaned Data): Validate, clean, and standardize the data
    #Gold (Business & ML Ready): Prepare data for analysis and ML

In [0]:
#Create a managed volume for raw dataset files
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sudoku_bronze.raw_files
""")

print("Volume created or verified successfully.")

In [0]:
#List available volumes in the Bronze schema
spark.sql("""
    SHOW VOLUMES IN workspace.sudoku_bronze
""").show(truncate=False)

In [0]:
#List files stored in the Bronze volume
volume_path = "/Volumes/workspace/sudoku_bronze/raw_files/"

display(dbutils.fs.ls(volume_path))

In [0]:
#Read the raw Sudoku dataset
file_path = "/Volumes/workspace/sudoku_bronze/raw_files/sudoku.csv"

sudoku_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false") #preserve the zeros that represent empty cells
    .csv(file_path)
)

#Inspect dataset structure
sudoku_df.printSchema()

#Preview the first records
display(sudoku_df.limit(5))

In [0]:
#Count the total number of records
total_records = sudoku_df.count()

print(f"Total records: {total_records:,}")

#Preview the first five records
display(sudoku_df.limit(5))